# verify08: 機能1「年齢との論理集合」検証（age_logic.py）

分岐で年齢（＋性別）を評価する機能の検証。ローカルで実行（BERTモデル不要・pure Python）。

- **Aタイプ**: 「N歳以上ですか？」→ 年齢から回答を導出（`derive_age_choice`）
- **Bタイプ**: router の `age>=16` 等（`eval_age_condition`）／「女性で12歳以上・男性65歳以上」等の論理集合（`eval_age_sex_gate`）／年齢で症候を絞る（`applicable_protocols`）
- 統合: `run_triage(age=...)` がAタイプ分岐を自動充填する

前提: `age_logic.py` / `triage_pipeline.py` / `transition_diagram/protocol.yaml` がこのノートと同じリポジトリにあること。

In [1]:
import importlib
import age_logic, triage_pipeline as tp
importlib.reload(age_logic); importlib.reload(tp)
g = tp.load_graph()
print('protocol.yaml ロード: ノード', len(g.index), '/ 症候', len(g.protocol_ids))

protocol.yaml ロード: ノード 129 / 症候 22


## 1. Bタイプ: `eval_age_condition`（router の age_condition）

In [2]:
cases = [
    ('age >= 16', 50, True), ('age >= 16', 10, False), ('age >= 16', None, False),
    ('age < 16', 10, True), ('age < 16', 50, False),
    ('age >= 16 OR age unknown', None, True), ('age >= 16 OR age unknown', 10, False),
]
for cond, a, exp in cases:
    got = age_logic.eval_age_condition(cond, a)
    print(f'  {cond:26s} age={str(a):4s} -> {got}  ', 'OK' if got==exp else f'NG(期待{exp})')
    assert got==exp, (cond,a)
print('eval_age_condition: 全ケースOK')

  age >= 16                  age=50   -> True   OK
  age >= 16                  age=10   -> False   OK
  age >= 16                  age=None -> False   OK
  age < 16                   age=10   -> True   OK
  age < 16                   age=50   -> False   OK
  age >= 16 OR age unknown   age=None -> True   OK
  age >= 16 OR age unknown   age=10   -> False   OK
eval_age_condition: 全ケースOK


## 2. Aタイプ: `derive_age_choice`（「N歳以上ですか？」を年齢から回答）

In [3]:
for nid, thr in [('chest_age_40',40),('common_cold_sweat_age_subquestion',40),('syncope_age_40_subquestion',40)]:
    n = g.node(nid)
    hi = age_logic.derive_age_choice(n, 70)   # 以上側
    lo = age_logic.derive_age_choice(n, 30)   # 未満側
    un = age_logic.derive_age_choice(n, None) # 不明→None
    print(f'  {nid:34s} 70->{hi}  30->{lo}  None->{un}')
    assert hi is not None and lo is not None and hi != lo and un is None
print('derive_age_choice: OK（以上/未満で別choice・不明はNone）')

  chest_age_40                       70->a  30->b  None->None
  common_cold_sweat_age_subquestion  70->i  30->ii  None->None
  syncope_age_40_subquestion         70->a  30->b  None->None
derive_age_choice: OK（以上/未満で別choice・不明はNone）


## 3. Bタイプ（論理集合）: `eval_age_sex_gate`

「(女性で12歳以上の場合、男性65歳以上の場合)」= (女∧age≥12) ∨ (男∧age≥65)

In [4]:
t = '(女性で12歳以上の場合、男性65歳以上の場合)'
for a, s, exp in [(70,'男',True),(70,'女',True),(20,'男',False),(20,'女',True),(None,'男',None)]:
    got = age_logic.eval_age_sex_gate(t, a, s)
    print(f'  age={str(a):4s} sex={s} -> 適用={got}  ', 'OK' if got==exp else f'NG(期待{exp})')
    assert got==exp
print('eval_age_sex_gate: 論理集合OK')

  age=70   sex=男 -> 適用=True   OK
  age=70   sex=女 -> 適用=True   OK
  age=20   sex=男 -> 適用=False   OK
  age=20   sex=女 -> 適用=True   OK
  age=None sex=男 -> 適用=None   OK
eval_age_sex_gate: 論理集合OK


## 4. 年齢で症候を絞る: `applicable_protocols`

In [5]:
routes = g.raw['chief_complaint_router']['routes']
ad = age_logic.applicable_protocols(routes, 70)
pe = age_logic.applicable_protocols(routes, 8)
print('  age=70 の pediatric系:', [p for p in ad if 'pediatric' in p], '（無いはず）')
print('  age=8  の pediatric系:', [p for p in pe if 'pediatric' in p])
assert not any('pediatric' in p for p in ad)
assert any('pediatric' in p for p in pe)
print('applicable_protocols: 年齢で小児/成人が正しく分岐 OK')

  age=70 の pediatric系: [] （無いはず）
  age=8  の pediatric系: ['pediatric_fever', 'pediatric_nausea_vomiting', 'pediatric_head_neck_trauma']
applicable_protocols: 年齢で小児/成人が正しく分岐 OK


## 5. 統合: `run_triage(age=...)` がAタイプ分岐を自動充填

In [6]:
common = {'intro_fire_or_emergency':'a','overview':'chief_complaint_classification','observation':'a',
          'common_breathing':'a','common_cold_sweat':'b','common_face_color':'b','common_conversation':'a'}
aug70 = age_logic.augment_answers_with_age(common, 70, g.index)
aug30 = age_logic.augment_answers_with_age(common, 30, g.index)
print('  age=70 追加:', {k:v for k,v in aug70.items() if k not in common})
print('  age=30 追加:', {k:v for k,v in aug30.items() if k not in common})
assert aug70['chest_age_40'] != aug30['chest_age_40']
print('run_triage 統合: 年齢で chest_age_40 等が自動で変わる OK')

  age=70 追加: {'common_cold_sweat_age_subquestion': 'i', 'syncope_age_40_subquestion': 'a', 'chest_age_40': 'a'}
  age=30 追加: {'common_cold_sweat_age_subquestion': 'ii', 'syncope_age_40_subquestion': 'b', 'chest_age_40': 'b'}
run_triage 統合: 年齢で chest_age_40 等が自動で変わる OK


## まとめ
- `eval_age_condition` / `derive_age_choice` / `eval_age_sex_gate` / `applicable_protocols` すべて期待通り。
- `run_triage(age=...)` は「N歳以上ですか？」型の分岐を年齢から自動で埋める（numeric age が権威・上書き）。
- 全 assert 通過なら機能1は検証OK。